Реализация Seed-and-Extend на питоне

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
def build_index(database, k):
    index = {}
    for i in range(len(database) - k + 1):
        kmer = database[i:i+k]
        if kmer not in index:
            index[kmer] = []
        index[kmer].append(i)
    return index

In [17]:
def find_seeds(query, index, k):

    seeds = []
    for i in range(len(query) - k + 1):
        kmer = query[i:i+k]
        if kmer in index:
            for j in index[kmer]:
                seeds.append((i, j))
    return seeds

In [18]:
def extend_alignment(query, database, seed_i, seed_j, k, match=3, mismatch=-3, gap=-2, X=2):
    initial_score = k * match
    max_score = initial_score
    max_pos_left = seed_i
    max_pos_right = seed_i + k - 1

    print(f"\n  Расширение влево от семени (позиции {seed_i}:{seed_j})")
    print(f"  Начальный вес семени: {initial_score}")

    left_align_q = []
    left_align_d = []
    cur_score = initial_score
    best_left_score = initial_score
    best_left_len = 0

    i, j = seed_i - 1, seed_j - 1
    step = 1

    while i >= 0 and j >= 0:
        match_score = match if query[i] == database[j] else mismatch

        diag = cur_score + match_score
        up = cur_score + gap
        left = cur_score + gap

        new_score = max(diag, up, left, 0)

        print(f"    Шаг {step}: позиции ({i},{j})")
        print(f"      query[{i}]={query[i]}, database[{j}]={database[j]}")
        print(f"      diag={diag}, up={up}, left={left} -> new_score={new_score}")

        if new_score > best_left_score:
            best_left_score = new_score
            best_left_len = step
            print(f"      → Новый максимум: {best_left_score}")

        if max_score - new_score >= X:
            print(f"      X-drop сработал: {max_score} - {new_score} = {max_score - new_score} >= {X}")
            break

        cur_score = new_score
        left_align_q.append(query[i])
        left_align_d.append(database[j])
        i -= 1
        j -= 1
        step += 1

    print(f"\n  Расширение вправо от семени")
    right_align_q = []
    right_align_d = []
    cur_score = initial_score
    best_right_score = initial_score
    best_right_len = 0

    i, j = seed_i + k, seed_j + k
    step = 1

    while i < len(query) and j < len(database):
        match_score = match if query[i] == database[j] else mismatch

        diag = cur_score + match_score
        up = cur_score + gap
        left = cur_score + gap

        new_score = max(diag, up, left, 0)
        print(f"    Шаг {step}: позиции ({i},{j})")
        print(f"      query[{i}]={query[i]}, database[{j}]={database[j]}")
        print(f"      diag={diag}, up={up}, left={left} -> new_score={new_score}")

        if new_score > best_right_score:
            best_right_score = new_score
            best_right_len = step
            print(f"      → Новый максимум: {best_right_score}")

        if max_score - new_score >= X:
            print(f"      X-drop сработал: {max_score} - {new_score} = {max_score - new_score} >= {X}")
            break

        cur_score = new_score
        right_align_q.append(query[i])
        right_align_d.append(database[j])
        i += 1
        j += 1
        step += 1

    left_align_q.reverse()
    left_align_d.reverse()

    full_align_q = left_align_q + [query[seed_i:seed_i+k]] + right_align_q
    full_align_d = left_align_d + [database[seed_j:seed_j+k]] + right_align_d

    return {
        'left_score': best_left_score,
        'right_score': best_right_score,
        'total_score': best_left_score + best_right_score - initial_score,
        'left_align_q': ''.join(left_align_q),
        'left_align_d': ''.join(left_align_d),
        'seed': query[seed_i:seed_i+k],
        'right_align_q': ''.join(right_align_q),
        'right_align_d': ''.join(right_align_d),
        'full_align_q': ''.join(full_align_q),
        'full_align_d': ''.join(full_align_d),
        'seed_pos_query': seed_i,
        'seed_pos_db': seed_j
    }

In [19]:
def seed_and_extend(query, database, k=4, X=2):
    print(f"Query (запрос):    {query}")
    print(f"Database (референс): {database}")
    print(f"Параметры: k={k}, X={X}")

    index = build_index(database, k)

    print(f"Индекс БД (k={k}-меры):")
    for kmer, positions in sorted(index.items()):
        print(f"  {kmer}: {positions}")

    query_kmers = [query[i:i+k] for i in range(len(query) - k + 1)]
    print(f"k-меры запроса: {query_kmers}")

    seeds = find_seeds(query, index, k)
    print(f"Найденные семена (позиция в query, позиция в database): {seeds}")
    if not seeds:
        return None

    results = []
    for seed_i, seed_j in seeds:
        print(f"\n>>> Анализ семени: query[{seed_i}:{seed_i+k}] = '{query[seed_i:seed_i+k]}' "
              f"совпадает с database[{seed_j}:{seed_j+k}] = '{database[seed_j:seed_j+k]}'")

        result = extend_alignment(query, database, seed_i, seed_j, k, X=X)
        results.append(result)

        print(f"\n  Результаты для семени:")
        print(f"    Левое расширение (max score: {result['left_score']}):")
        print(f"      {result['left_align_q']}[{result['seed']}]{result['right_align_q']}")
        print(f"      {result['left_align_d']}[{result['seed']}]{result['right_align_d']}")
        print(f"    Полное выравнивание:")
        print(f"      {result['full_align_q']}")
        print(f"      {result['full_align_d']}")
        print(f"    Итоговый Smax: {result['total_score']}")

    best_result = max(results, key=lambda x: x['total_score'])

    print(f"Семя: '{best_result['seed']}' (позиции: query[{best_result['seed_pos_query']}], "
          f"database[{best_result['seed_pos_db']}])")
    print(f"Итоговый Smax: {best_result['total_score']}")
    print(f"Левое расширение (score: {best_result['left_score']}):")
    print(f"  {best_result['left_align_q']}")
    print(f"  {best_result['left_align_d']}")
    print(f"Правое расширение (score: {best_result['right_score']}):")
    print(f"  {best_result['right_align_q']}")
    print(f"  {best_result['right_align_d']}")
    print(f"Полное выравнивание:")
    print(f"  {best_result['full_align_q']}")
    print(f"  {best_result['full_align_d']}")

    return best_result

In [20]:
query = "GGATCCATCTATA"
database = "CTAGGATCCAGGCTACTACA"

result = seed_and_extend(query, database, k=4, X=2)

Query (запрос):    GGATCCATCTATA
Database (референс): CTAGGATCCAGGCTACTACA
Параметры: k=4, X=2
Индекс БД (k=4-меры):
  ACTA: [14]
  AGGA: [2]
  AGGC: [9]
  ATCC: [5]
  CAGG: [8]
  CCAG: [7]
  CTAC: [12, 15]
  CTAG: [0]
  GATC: [4]
  GCTA: [11]
  GGAT: [3]
  GGCT: [10]
  TACA: [16]
  TACT: [13]
  TAGG: [1]
  TCCA: [6]
k-меры запроса: ['GGAT', 'GATC', 'ATCC', 'TCCA', 'CCAT', 'CATC', 'ATCT', 'TCTA', 'CTAT', 'TATA']
Найденные семена (позиция в query, позиция в database): [(0, 3), (1, 4), (2, 5), (3, 6)]

>>> Анализ семени: query[0:4] = 'GGAT' совпадает с database[3:7] = 'GGAT'

  Расширение влево от семени (позиции 0:3)
  Начальный вес семени: 12

  Расширение вправо от семени
    Шаг 1: позиции (4,7)
      query[4]=C, database[7]=C
      diag=15, up=10, left=10 -> new_score=15
      → Новый максимум: 15
    Шаг 2: позиции (5,8)
      query[5]=C, database[8]=C
      diag=18, up=13, left=13 -> new_score=18
      → Новый максимум: 18
    Шаг 3: позиции (6,9)
      query[6]=A, database[9]=A
 